# Libraries

In [1]:
from obj import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy

# Beam dataset

Just one sample was select for the test.

In [2]:
file_path = r'final_df_tempo_limite.xlsx'
full_df = pd.read_excel(file_path)
df = full_df.head(1)
df

,b_w,h,f_ck,f_yk,m_rd,m_gk,m_qk,a_s,time
0,0.14,0.3,20000,500000,9.836582,6.323517,0.702613,0.000076,50.621397


# State Limit Function evaluation

### Simulation parameters

In [ ]:
# Number of simulations
n_simul = 100

# Time lapse for the durability analysis (in years)
times = np.arange(0, 101, 1)

# Número de simulações Monte Carlo
n_simul = 100

# Tempos de análise (anos)
times = np.arange(0, 101, 10)

# Cobrimento das armaduras (mm)
cob = 25

# Módulo de elasticidade do aço (kPa)
e_s = 200E6

# Progressão da carbonatação (mm/ano^0.5)
prog_y = 6

# Stochastic simulation

### Initialization Variables

In [ ]:
results_bw = []
results_mr = []
results_ms = []
results_as = []   
results_g  = []     
t_without_cor  = []   

### Monte Carlo simulation

In [ ]:
for i, row in df.iterrows():
    # Beam parameters
    b_w = row['b_w']            
    h = row['h']                
    f_ck = row['f_ck']          
    f_yk = row['f_yk']                
    m_gk = row['m_gk']         
    m_qk = row['m_qk']          
    a_s_initial = row['a_s']

    # Stochastic simulation   
    for j in range(n_simul):
        m_r_list = []             
        a_s_list = []               
        m_s_list = []               
        g_list = []                 

        # Sampling the random variables
        f_ck_sample = np.random.normal(1.22 * f_ck, 0.12 * 1.22 * f_ck)
        f_yk_sample = np.random.normal(1.22 * f_yk, 0.05 * 1.22 * f_yk)
        m_gk_sample = np.random.normal(1.06 * m_gk, 0.12 * 1.06 * m_gk)
        relacao_d_h = np.random.uniform(0.8, 0.9)
        i_corr_20 = np.random.normal(loc=0.431, scale=0.259)
        if i_corr_20 < 0:
            i_corr_20 = 1E-5
        else:
            i_corr_20 = i_corr_20

        # Loop through each time step (e.g., years)
        for k in times:
            # Sampling the stochastic random variables for each time step
            m_qk_sample = np.random.gumbel(0.21 * m_qk, 0.21 * 0.76 * m_qk)
            temp = np.random.uniform(20, 30)
            d_barras = 12.5/1000
            n_barras = a_s_initial / (np.pi * d_barras**2 / 4)
            y_carb = prog_y * k**0.5

            # Bending moment capacity calculation
            if y_carb > cob:
               t_dur = k - t_without_cor[-1]
               m_rd, c_f, d_novo, d = momento_resistente_com_corrosao_azad_algohi(d_barras, n_barras, f_ck_sample, f_yk_sample, e_s, b_w, h, relacao_d_h, i_corr_20, temp, k, t_without_cor[-1])
            else:
                t_without_cor.append(k)
                m_rd = momento_limite_armadura_simples(a_s_initial, b_w, h, relacao_d_h, f_ck_sample, f_yk_sample, e_s)

            # Armazenar os resultados para este passo de tempo
            m_r_list.append(m_rd)
            m_s_list.append(m_gk_sample + m_qk_sample)
            # a_s_list.append(a_s_degraded)
            g_list.append(m_rd - (m_gk_sample + m_qk_sample))

        # Resultados da simulação temporal completa
        results_mr.append(m_r_list)
        results_ms.append(m_s_list)
        results_as.append(a_s_list)
        results_g.append(g_list)
        h_aux = [h] * len(times)
        f_ck_aux = [f_ck_sample] * len(times)
        f_yk_aux = [f_yk_sample] * len(times)

In [11]:
np.array(results_g).flatten().tolist()

[3.478719219465834,
 3.768938814274649,
 3.347655501768741,
 3.086781983878846,
 2.627368143499311,
 1.399792292999825,
 0.9750146787419212,
 0.980871160660441,
 0.9251498905152244,
 0.33291462110713077,
 -0.0382589718584887,
 2.7646360400290577,
 2.7229211401759335,
 2.3718284898895554,
 0.49021421472939153,
 0.13370301677555574,
 0.10588979345989191,
 -1.5048959179690717,
 -1.8483633111231024,
 -2.0317658486018484,
 -1.9169870821592578,
 -2.4937011463353764,
 3.0003659124220503,
 2.9723809882553143,
 2.7290046171826736,
 2.776241931011471,
 2.3541944516434192,
 1.3766800393448007,
 1.5929618797050162,
 1.1417479554986354,
 0.48616469270924867,
 -0.10409881383072506,
 -0.07487364065493907,
 4.170288586553788,
 4.234606053006457,
 4.026381223280602,
 4.061789544255372,
 4.176406697288216,
 4.107357645488713,
 4.013918426438039,
 3.9586006671114786,
 3.827600067099917,
 3.954410107959787,
 3.8141234023291837,
 -0.31491218739505733,
 -0.1752644483699921,
 -0.4751019715544693,
 -1.3500925